In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import random

# some calculations are sloooow. set this to True to reduce samples width to 
# increase speed, when you only care about the later calculation
speedup = False

# the path of the image to be analyzed
path = "03.png"

# choose which paths to use, by uncommenting the according line

# load images and show
print()
print(f"(A)")

S = mpimg.imread(path)
H, W = S.shape
N = H * W

if S.ndim == 3:
    S = S.mean(axis=2)
    
plt.imshow(S, cmap="gray")
plt.title("$S(x, y)$")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

print(f"min: {S.min()} max: {S.max()} mean: {S.mean()}")

# zero mean
print()
print(f"(B)")

hat_S = (S/N).sum()
S = S - hat_S

print(f"min: {S.min()} max: {S.max()} mean: {S.mean()}")

# correlation between neighboring pixels
print()
print(f"(C)")

steps = 1
if speedup:
    steps = 32

d_values = np.arange(0, W, steps)
y_values = []

for i, d in enumerate(d_values):
    print(f"progress... {i + 1}/{len(d_values)}")
    
    temp = np.zeros((H, W - d))
    H_, W_ = temp.shape
    N_ = H_ * W_
    
    for y in range(H):
        for x in range(W - d):
            temp[y, x] = S[y, x] * S[y, x + d] / N_
    
    RS = temp.sum()
    #print(f"$R^S({d}) = {RS}$")
    y_values.append(RS)
    
plt.plot(d_values, y_values)
plt.title("correlation of neighboring pixels")
plt.xlabel("distance $d$")
plt.ylabel("correlation")
plt.show()

# furiour power specturm 1-D
print()
print(f"(D)")

steps = 1
if speedup:
    steps = 16

L = np.arange(0, W, steps)
k_values = L * 2 * np.pi / W

SC = np.zeros(len(k_values))
SS = np.zeros(len(k_values))
S2 = np.zeros(len(k_values))

for i, k in enumerate(k_values):
    print(f"progress... {i + 1}/{len(k_values)}")
    
    sum_c = 0
    sum_s = 0
    
    for y in range(H):
        for x in range(W):
            sum_c += S[y, x] * np.cos(k * x);
            sum_s += S[y, x] * np.sin(k * x);
    
    SC[i] = sum_c
    SS[i] = sum_s
    S2[i] = sum_c * sum_c + sum_s * sum_s

plt.plot(k_values, S2)
plt.title("Fourier power spectrum (1D)")
plt.xlabel("Fourier frequency $k$")
plt.ylabel("signal power $|S(k)|^2$")
plt.xscale('log')
plt.yscale('log')
plt.show()

# furiour power specturm 2-D
print()
print(f"(E)")

plt.imshow(S, cmap="gray")
plt.title("original image $S$")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

SFurier = np.fft.fft2(S)

kx = np.fft.fftshift(np.fft.fftfreq(W)) * W
ky = np.fft.fftshift(np.fft.fftfreq(H)) * H
im = plt.imshow(
    np.log(np.abs(np.fft.fftshift(SFurier))),
    extent=[kx[0], kx[-1], ky[0], ky[-1]],
)
plt.title("Fourier components $\log |\mathcal{S}(k)|$")
plt.xlabel("$k_x$ (cycles/image)")
plt.ylabel("$k_y$ (cycles/image)")
plt.colorbar(im)
plt.show()

S_ = np.real(np.fft.ifft2(SFurier))

plt.imshow(S_, cmap="gray")
plt.title("$S'$ after the inverse Fourier transform of the Fourier components")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

print(f"sanity check: the difference should be 0")
Sdiff = S - S_
im = plt.imshow(Sdiff)
plt.title("$S - S'$")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar(im)
plt.show()

# matlab code in excercise translated to python
#kx_1d = np.fft.fftshift(np.fft.fftfreq(W)) * W
#ky_1d = np.fft.fftshift(np.fft.fftfreq(H)) * H
#
#kx, ky = np.meshgrid(kx_1d, ky_1d)
#k = np.sqrt(kx**2 + ky**2)
#
#fig, axes = plt.subplots(1, 3, figsize=(12, 4))
#
#im0 = axes[0].imshow(kx)
#axes[0].set_title("$k_x$")
#axes[0].axis("off")
#plt.colorbar(im0, ax=axes[0])
#
#im1 = axes[1].imshow(ky)
#axes[1].set_title("$k_y$")
#axes[1].axis("off")
#plt.colorbar(im1, ax=axes[1])
#
#im2 = axes[2].imshow(k)
#axes[2].set_title("$|k|$")
#axes[2].axis("off")
#plt.colorbar(im2, ax=axes[2])
#
#plt.tight_layout()
#plt.show()

#####################
print()
print("reached end!")